# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib python-dotenv

import os, getpass
import duckdb
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
DECISION_DATE = "2026-03-15"  # same Lane 2 slice as w03_data_contract.ipynb: days 1-15 known, 16-31 outcome

fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

print(f"Connected. Reusing month={MONTH}, decision_date={DECISION_DATE} from w03_data_contract.ipynb.")

In [ ]:
raw = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= DATE '{DECISION_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_prev15,
            SUM(CASE WHEN report_date <= DATE '{DECISION_DATE}' THEN gsc_clicks ELSE 0 END)      AS clicks_prev15,
            AVG(CASE WHEN report_date <= DATE '{DECISION_DATE}' THEN gsc_avg_position END)       AS avg_position_prev15,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{DECISION_DATE}' AND gsc_impressions > 0
                                 THEN report_date END)                                            AS active_days_prev15,
            SUM(CASE WHEN report_date > DATE '{DECISION_DATE}' THEN gsc_impressions ELSE 0 END)  AS imp_last15
        FROM {fact_month}
        GROUP BY 1, 2
        HAVING imp_prev15 >= 10
    )
    SELECT
        d.*,
        c.content_type, c.search_volume, c.competition, c.competition_level, c.cpc, c.main_intent,
        c.word_count, c.char_count, c.provider_used, c.model_used,
        c.content_created_date, c.content_updated_date,
        c.last_optimized_date, c.backlinks, c.is_published, c.is_deleted
    FROM daily d
    JOIN {dim_content} c USING (client_hash_id, content_hash_id)
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()

print(f"{len(raw):,} live (published, not deleted) content items with enough prev-window history")
raw.head()

In [ ]:
df = raw.copy()

# Keyword-level fields only exist for keyword-sourced content -- missingness follows
# content_type (feedly article ~ 100% missing), same pattern as the starter CSV.
# A blind fillna(0) would silently encode content_type into the features -- flag instead.
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["content_type"] = df["content_type"].fillna("unknown")
df["main_intent"] = df["main_intent"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

# content_created_date is always <= the decision moment for anything with prev-window
# history, so content_age_days_at_decision is safe as-is.
decision_date = pd.Timestamp(DECISION_DATE)
df["content_age_days_at_decision"] = (decision_date - pd.to_datetime(df["content_created_date"])).dt.days

# content_updated_date is NOT safe as-is -- see the leakage hunt below. This is the
# clipped, honest version: "no update observed yet" if the update postdates the decision.
updated = pd.to_datetime(df["content_updated_date"])
safe_update = updated.where(updated <= decision_date)
df["has_update_before_decision"] = safe_update.notna().astype(int)
df["days_since_update_at_decision"] = (decision_date - safe_update).dt.days
df["days_since_update_at_decision"] = df["days_since_update_at_decision"].fillna(
    df["content_age_days_at_decision"]  # never updated (that we can see) = as old as the page itself
)

df["ctr_prev15"] = df["clicks_prev15"] / df["imp_prev15"].replace(0, np.nan)
df["is_declining"] = (df["imp_last15"] < 0.8 * df["imp_prev15"]).astype(int)

print(f"{len(df):,} rows, {df['has_keyword_data'].mean():.1%} have keyword data, "
      f"{df['has_update_before_decision'].mean():.1%} show an update before the decision moment")
df[["content_age_days_at_decision", "days_since_update_at_decision", "has_keyword_data", "is_declining"]].describe()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Type | Available before the decision moment (Mar 16)? |
|---|---|---|---|---|
| `imp_prev15` | GSC impressions, days 1–15 | none needed (`HAVING imp_prev15 >= 10` already filters blanks) | numeric | Yes — built only from `report_date <= Mar 15` |
| `clicks_prev15` | GSC clicks, days 1–15 | none needed | numeric | Yes |
| `ctr_prev15` | `clicks_prev15 / imp_prev15` | can't be null post-filter | numeric | Yes |
| `avg_position_prev15` | mean GSC position, days 1–15 | none needed | numeric | Yes |
| `active_days_prev15` | days with impressions > 0, days 1–15 | none needed | numeric | Yes |
| `search_volume`, `competition`, `cpc` | keyword-market metrics from `dim_content` | blank for 0–100% of rows depending on `content_type` (feedly article ≈ 100% missing) — flagged with `has_keyword_data`, never filled with 0 | numeric | Keyword-market data doesn't depend on this page's outcome, but `dim_content` isn't date-versioned (see §4) |
| `competition_level`, `content_type`, `main_intent` | categorical keyword/content context | blank → `"unknown"` category | categorical | Same caveat as above |
| `word_count`, `char_count` | article length | blank → flagged with `has_word_count`, not filled with 0 | numeric | Yes — fixed at creation |
| `content_age_days_at_decision` | `decision_date − content_created_date` | none — every row has a creation date | numeric | Yes, by construction |
| `days_since_update_at_decision` | `decision_date − content_updated_date`, **clipped** to ignore updates after the decision | falls back to `content_age_days_at_decision` when no update precedes the decision | numeric | Yes — see the leakage hunt below for why the *unclipped* version is not |
| `has_keyword_data`, `has_word_count`, `has_update_before_decision` | missingness/availability flags | n/a | boolean | Yes |

`imp_last15` is also pulled (to build the label) but is **not** a feature — it's the deliberate leak tested in section 3.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Test 1 -- future/overlapping window: is raw content_updated_date safe to use as-is?
naive_future = (pd.to_datetime(df["content_updated_date"]) > decision_date).sum()
print(f"{naive_future:,} of {len(df):,} rows in this modeling set "
      f"({naive_future / len(df):.1%}) have a content_updated_date AFTER the decision moment "
      f"({DECISION_DATE}).")
print(
    "A naive 'days since update' built straight from this column would hand most rows a "
    "future timestamp -- a page refreshed in April tells you something happened to it, "
    "information a March 16 reviewer could never have had. That's why "
    "`days_since_update_at_decision` in section 1 is CLIPPED: any update after the decision "
    "moment is treated as 'no update observed yet', not as a peek forward."
)

In [ ]:
# Test 2 -- product/decision flag: does "already optimized" just encode a past decision?
opt_check = (
    df.assign(was_optimized=df["last_optimized_date"].notna())
    .groupby("was_optimized")["is_declining"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "decline_rate"})
)
print(opt_check)
print(
    "\nIf decline_rate differs a lot between the two rows above, `last_optimized_date` isn't "
    "a raw world signal -- it's correlated with a human/product decision already made about "
    "this exact page (someone flagged or fixed it before). That circularity is why it's "
    "excluded in section 4, kept only as a possible baseline to compare against."
)

In [ ]:
# Test 3 -- label-derived feature: add imp_last15 (used to build is_declining) on purpose
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

honest_numeric = [
    "imp_prev15", "clicks_prev15", "ctr_prev15", "avg_position_prev15", "active_days_prev15",
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days_at_decision", "days_since_update_at_decision",
    "has_keyword_data", "has_word_count", "has_update_before_decision",
]
categorical = ["content_type", "competition_level", "main_intent"]

def quick_auc(model_df, numeric_cols, categorical_cols, label_col="is_declining", seed=42):
    data = model_df.dropna(subset=[label_col]).copy()
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())
    X, y = data[numeric_cols + categorical_cols], data[label_col]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    pre = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    return roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])

base_rate = df["is_declining"].mean()
print(f"base rate (share actually declining): {base_rate:.3f}  -- every AUC below sits next to this")

honest_auc = quick_auc(df, honest_numeric, categorical)
leaky_auc = quick_auc(df, honest_numeric + ["imp_last15"], categorical)

print(f"honest AUC ({len(honest_numeric)} features, no leak): {honest_auc:.3f}")
print(f"leaky AUC (+ imp_last15, the label leak):  {leaky_auc:.3f}  <- jumps toward 1.0, the same confession as w03_data_contract")
print(f"\nKEEPING the honest number: {honest_auc:.3f}. imp_last15 is not part of the kept feature set.")

In [ ]:
# Test 4 -- honest split: random split lets a client's own pages leak into both halves.
# Client-grouped split is the honest question: does this generalize to a client never seen?
def quick_auc_grouped(model_df, numeric_cols, categorical_cols, label_col="is_declining", seed=42):
    data = model_df.dropna(subset=[label_col]).copy()
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())
    X = data[numeric_cols + categorical_cols]
    y, groups = data[label_col], data["client_hash_id"]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr_idx, te_idx = next(gss.split(X, y, groups))
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    pre = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    return roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])

grouped_auc = quick_auc_grouped(df, honest_numeric, categorical)

print(f"random-split honest AUC:   {honest_auc:.3f}  (a client's pages can land in both halves)")
print(f"client-grouped honest AUC: {grouped_auc:.3f}  (no client's pages appear in both halves)")
print(f"gap: {honest_auc - grouped_auc:+.3f}  -- the size of this gap is itself a finding about how much the random split let the model memorize per-client quirks")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `provider_used`, `model_used` | Generation-process artifacts, not performance signals — same rule as the starter CSV's data dictionary. |
| `backlinks` | `dim_content` is a dimension table, not date-versioned — I can't confirm this reflects March 15 rather than whatever date the release was built. ~50% missing besides. Excluding until I can pin down its "as-of" date. |
| `last_optimized_date`, `optimization_eligible_date` | Product/decision flags — Test 2 below shows `last_optimized_date` correlates with `is_declining`, meaning it partly encodes a past human decision to optimize *this exact page*, not a raw world signal. Per the leakage taxonomy: fine as a baseline to compare against, never as a model input. |
| raw `content_updated_date` | Future/overlapping-window leak — Test 1 below shows most rows have an update timestamp *after* the decision moment. Only the clipped `days_since_update_at_decision` is kept. |
| `is_deleted`, `is_published` | Used as a row filter upstream (only live, published pages are review candidates) — not model inputs. |
| `keyword_hash_id`, `url_hash_id` | Extra identifiers — context/joins only, same rule as `content_hash_id`. |
| `imp_last15` | The deliberate label-derived leak (Test 3 below) — proven and removed. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.